# Building Agentic AutoML

## Goal

This notebook is a hands-on journey to build an Agentic AutoML system from scratch.

Each version introduces one new concept, allowing the agent to evolve step by step while practicing AutoML and agent development.

---

## Version 2

In this version, we extend the agent with a dataset inspection step.

Before training, the agent analyzes the dataset structure, including its dimensions, target distribution, feature types, missing values, and feature cardinality.

The rest of the pipeline remains intentionally simple, using the same task detection, preprocessing, LightGBM baseline, train-validation split, evaluation, and state-based architecture introduced in Version 1.

## 1. Imports

In [1]:
from pathlib import Path

import pandas as pd

from lightgbm import LGBMClassifier, LGBMRegressor
from sklearn.metrics import mean_squared_error, roc_auc_score
from sklearn.model_selection import train_test_split

## 2. Data

We now use a more realistic tabular dataset: Adult Income.

The target column is provided by the user.

In [2]:
DATA_PATH = "/kaggle/input/datasets/lucalullo/agentic-automl-datasets/adult_income.csv"
TARGET = "income"

df = pd.read_csv(DATA_PATH)

df.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


## 3. Task Detection

The agent first inspects the target to identify the machine learning task.

For now, we use a simple rule based on the number of unique target values.

In [3]:
def detect_task(df, target):
    n_unique = df[target].nunique()

    if n_unique <= 20:
        return "classification"

    return "regression"

In [4]:
task = detect_task(df, TARGET)

task

'classification'

## 4. Feature Detection

The agent identifies numerical and categorical features.

For now, feature types are detected directly from the dataframe dtypes.

In [5]:
def detect_features(df, target):
    X = df.drop(columns=target)

    numerical = X.select_dtypes(include="number").columns.tolist()
    categorical = X.select_dtypes(exclude="number").columns.tolist()

    return numerical, categorical

In [6]:
numerical_features, categorical_features = detect_features(df, TARGET)

numerical_features, categorical_features

(['age',
  'fnlwgt',
  'education_num',
  'capital_gain',
  'capital_loss',
  'hours_per_week'],
 ['workclass',
  'education',
  'marital_status',
  'occupation',
  'relationship',
  'race',
  'sex',
  'native_country'])

## 5. Data Inspection

The agent now inspects the dataset before training.

It summarizes the dataset dimensions, target distribution, feature data types, missing values, and feature cardinality.

In [7]:
def inspect_dataset(df, target):
    X = df.drop(columns=target)
    y = df[target]

    target_info = {
        "name": target,
        "dtype": str(y.dtype),
        "unique_values": int(y.nunique())
    }

    if y.nunique() <= 20:
        target_info["distribution"] = y.value_counts(dropna=False).to_dict()
    else:
        target_info["summary"] = y.describe().to_dict()

    return {
        "shape": {"rows": len(df), "columns": len(df.columns)},
        "target": target_info,
        "dtypes": {column: str(dtype) for column, dtype in X.dtypes.items()},
        "missing_values": df.isna().sum().to_dict(),
        "cardinality": X.nunique(dropna=True).to_dict()
    }

In [8]:
inspection = inspect_dataset(df, TARGET)

inspection

{'shape': {'rows': 48842, 'columns': 15},
 'target': {'name': 'income',
  'dtype': 'object',
  'unique_values': 2,
  'distribution': {'<=50K': 37155, '>50K': 11687}},
 'dtypes': {'age': 'int64',
  'workclass': 'object',
  'fnlwgt': 'int64',
  'education': 'object',
  'education_num': 'int64',
  'marital_status': 'object',
  'occupation': 'object',
  'relationship': 'object',
  'race': 'object',
  'sex': 'object',
  'capital_gain': 'int64',
  'capital_loss': 'int64',
  'hours_per_week': 'int64',
  'native_country': 'object'},
 'missing_values': {'age': 0,
  'workclass': 2799,
  'fnlwgt': 0,
  'education': 0,
  'education_num': 0,
  'marital_status': 0,
  'occupation': 2809,
  'relationship': 0,
  'race': 0,
  'sex': 0,
  'capital_gain': 0,
  'capital_loss': 0,
  'hours_per_week': 0,
  'native_country': 857,
  'income': 0},
 'cardinality': {'age': 74,
  'workclass': 8,
  'fnlwgt': 28523,
  'education': 16,
  'education_num': 16,
  'marital_status': 7,
  'occupation': 14,
  'relationship'

## 6. Data Preparation

LightGBM can handle categorical features directly.

The agent converts categorical columns to the pandas `category` dtype and separates features from the target.

In [9]:
def prepare_data(df, target, categorical_features):
    data = df.copy()

    for column in categorical_features:
        data[column] = data[column].astype("category")

    X = data.drop(columns=target)
    y = data[target]

    return X, y

In [10]:
X, y = prepare_data(df, TARGET, categorical_features)

X.dtypes

age                  int64
workclass         category
fnlwgt               int64
education         category
education_num        int64
marital_status    category
occupation        category
relationship      category
race              category
sex               category
capital_gain         int64
capital_loss         int64
hours_per_week       int64
native_country    category
dtype: object

## 7. Baseline Model

The agent selects a LightGBM model according to the detected task.

For now, we use the default model configuration.

In [11]:
def select_model(task):
    if task == "classification":
        return LGBMClassifier(random_state=42, verbosity=-1)

    return LGBMRegressor(random_state=42, verbosity=-1)

In [12]:
model = select_model(task)

model

LGBMClassifier(random_state=42, verbosity=-1)

## 8. Metric Selection

The agent selects a simple evaluation metric according to the detected task.

For now, we use ROC AUC for classification and RMSE for regression.

In [13]:
def select_metric(task):
    if task == "classification":
        return "roc_auc"

    return "rmse"

In [14]:
metric = select_metric(task)

metric

'roc_auc'

## 9. Training and Evaluation

The agent splits the dataset into training and validation sets, trains the selected model, and evaluates its performance.

For now, we use a single train-validation split.

In [15]:
def train_and_evaluate(X, y, model, task):
    stratify = y if task == "classification" else None

    X_train, X_valid, y_train, y_valid = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=stratify
    )

    model.fit(X_train, y_train)

    if task == "classification":
        predictions = model.predict_proba(X_valid)[:, 1]
        score = roc_auc_score(y_valid, predictions)
    else:
        predictions = model.predict(X_valid)
        score = mean_squared_error(y_valid, predictions) ** 0.5

    return score

In [16]:
score = train_and_evaluate(X, y, model, task)

score

np.float64(0.9311658417981501)

## 10. Agent

We now combine the previous components into a single agent.

The agent receives a dataset and a target column, inspects the dataset, makes the required decisions, trains the baseline model, and returns its state.

In [17]:
def agent(data_path, target):
    df = pd.read_csv(data_path)

    task = detect_task(df, target)
    numerical, categorical = detect_features(df, target)
    inspection = inspect_dataset(df, target)
    X, y = prepare_data(df, target, categorical)

    model = select_model(task)
    metric = select_metric(task)
    score = train_and_evaluate(X, y, model, task)

    return {
        "task": task,
        "numerical_features": numerical,
        "categorical_features": categorical,
        "inspection": inspection,
        "model": model.__class__.__name__,
        "metric": metric,
        "score": float(score)
    }

In [18]:
state = agent(DATA_PATH, TARGET)

state

{'task': 'classification',
 'numerical_features': ['age',
  'fnlwgt',
  'education_num',
  'capital_gain',
  'capital_loss',
  'hours_per_week'],
 'categorical_features': ['workclass',
  'education',
  'marital_status',
  'occupation',
  'relationship',
  'race',
  'sex',
  'native_country'],
 'inspection': {'shape': {'rows': 48842, 'columns': 15},
  'target': {'name': 'income',
   'dtype': 'object',
   'unique_values': 2,
   'distribution': {'<=50K': 37155, '>50K': 11687}},
  'dtypes': {'age': 'int64',
   'workclass': 'object',
   'fnlwgt': 'int64',
   'education': 'object',
   'education_num': 'int64',
   'marital_status': 'object',
   'occupation': 'object',
   'relationship': 'object',
   'race': 'object',
   'sex': 'object',
   'capital_gain': 'int64',
   'capital_loss': 'int64',
   'hours_per_week': 'int64',
   'native_country': 'object'},
  'missing_values': {'age': 0,
   'workclass': 2799,
   'fnlwgt': 0,
   'education': 0,
   'education_num': 0,
   'marital_status': 0,
   'occ

## 11. Regression Test

The same agent should also work with a regression dataset.

We only change the input dataset and target column. The agent must detect the new task and adapt automatically.

In [19]:
REGRESSION_PATH = "/kaggle/input/datasets/lucalullo/agentic-automl-datasets/simple_regression.csv"

regression_state = agent(REGRESSION_PATH, "target")

regression_state

{'task': 'regression',
 'numerical_features': ['num_1', 'num_2', 'num_3', 'num_4'],
 'categorical_features': ['category_1', 'category_2'],
 'inspection': {'shape': {'rows': 1200, 'columns': 7},
  'target': {'name': 'target',
   'dtype': 'float64',
   'unique_values': 1200,
   'summary': {'count': 1200.0,
    'mean': 1.4709987021728987,
    'std': 5.814924591631912,
    'min': -16.008721903706515,
    '25%': -2.555178021015151,
    '50%': 1.3890592022588901,
    '75%': 5.322514631724257,
    'max': 24.027912326916383}},
  'dtypes': {'num_1': 'float64',
   'num_2': 'float64',
   'num_3': 'float64',
   'num_4': 'float64',
   'category_1': 'object',
   'category_2': 'object'},
  'missing_values': {'num_1': 9,
   'num_2': 8,
   'num_3': 10,
   'num_4': 4,
   'category_1': 9,
   'category_2': 20,
   'target': 0},
  'cardinality': {'num_1': 1191,
   'num_2': 1192,
   'num_3': 1190,
   'num_4': 1196,
   'category_1': 3,
   'category_2': 2}},
 'model': 'LGBMRegressor',
 'metric': 'rmse',
 'scor

## Notes

The agent can now:

- detect whether the task is classification or regression
- identify numerical and categorical features
- inspect dataset dimensions
- inspect target distribution or summary statistics
- inspect feature data types
- detect missing values
- measure feature cardinality
- train a LightGBM baseline
- select an appropriate evaluation metric
- return the result together with the dataset inspection in its state

The architecture remains intentionally simple.

Future versions will introduce new components and gradually evolve the architecture.